# deepEmulator — DINO pretraining (Colab, Drive-resumed)

Self-supervised ViT-tiny pretraining on a multi-cartridge emulator-frame corpus. The output is a frozen encoder that maps any 96×96 grayscale emulator frame to a 256-dim latent, used by P15's DDQN-on-latents and P16's attention overlay.

**One-time Drive setup:**
```
MyDrive/deepEmulator/
  data/frames/<corpus_name>/index.json + chunk_*.npz   # collected via P12 FrameCollector
  encoders/                                            # created here on first save
```

If you don't have a corpus yet, run a small collection cell first (commented out below).

In [ ]:
!nvidia-smi -L || echo 'no GPU — runtime > change runtime type > GPU'

In [ ]:
!pip install -q git+https://github.com/juangarassino/deepEmulator.git

In [ ]:
# Optional: collect frames first (only run once per corpus_name)
# from google.colab import drive; drive.mount('/content/drive')
# from deepEmulator.data.frame_corpus import FrameStorage, FrameCollector
# from deepEmulator.platforms.gameboy import PyBoyEnv
# from deepEmulator.cartridges import pokemon_red  # register
# from deepEmulator.core import registry
# adapter = registry.get('POKEMON RED')(init_state='/content/drive/MyDrive/deepEmulator/states/init.state')
# env = PyBoyEnv(adapter, rom_path='/content/drive/MyDrive/deepEmulator/roms/PokemonRed.gb', headless=True)
# storage = FrameStorage('/content/drive/MyDrive/deepEmulator/data/frames/pokemon_red')
# coll = FrameCollector([env], storage)
# coll.run(50_000); storage.flush()

In [ ]:
# Pretrain DINO. Re-run this cell after a Colab disconnect — it auto-resumes
# from the latest encoder bundle under MyDrive/deepEmulator/encoders/dino/.
from deepEmulator.training.colab_pretrain import run_in_colab

run_in_colab(
    corpus='deepEmulator/data/frames/pokemon_red',
    steps=50_000,
    batch_size=64,
    save_every=2_000,
    out_dim=4096,
    n_local_crops=6,
    resume=True,
)

In [ ]:
# Inspect the loss curve
import pandas as pd, glob
from pathlib import Path
root = Path('/content/drive/MyDrive/deepEmulator/encoders/dino')
latest = (root / 'latest.txt').read_text().strip() if (root / 'latest.txt').exists() else sorted(glob.glob(str(root / '*')))[-1]
df = pd.read_csv(Path(latest) / 'metrics.tsv', sep='\t')
df.plot(x='Step', y='Loss', figsize=(10, 4))